# layout

> FastHTML constructors for stack, cluster, grid, and document flow layouts

In [ ]:
#| default_exp layout

Component styles set internal padding. Layout attributes on the parent control spacing between child elements. Each FastHTML constructor returns a `Div` with the matching HTML attribute:

- `Stack` sets `stack` for a vertical column.
- `Cluster` sets `cluster` for a wrapping row.
- `Grid` sets `grid` for adaptive equal-width columns.
- `Flow` sets `flow` for document spacing.

Put any of those attributes directly on a semantic element such as `Section(..., stack=True)` to match the same CSS rules without an extra wrapper.

In [ ]:
from fasthtml.common import Div, H2, H3, P, Code, Small
from answerui.registry import hypelt
from answerui.demo import example

In [ ]:
from fasthtml.common import Article, Header, Section, Ul, Li, to_xml
from answerui.demo import demo

## Stack

`Stack` arranges its children in a flex column and stretches them across the available width. Setting `gap` on the parent puts space only between adjacent children, avoiding a trailing margin after the last child. The default gap is `--space`.

Every layout constructor accepts CSS lengths or token references (such as `'var(--space-2)'`) for its sizing options. Layout rules stay in the stylesheet. `_layout` prepends those custom properties to any `style` string the caller passes:

In [ ]:
def _layout(*c, props, style='', **attrs):
    style = ''.join(f'{k}: {v};' for k,v in props.items() if v is not None) + (style or '')
    return Div(*c, style=style or None, **attrs)

In [ ]:
@hypelt(sel='[stack]')
def Stack(
    *c,       # Children to arrange vertically
    gap=None, # CSS length or token reference; None uses --space
    **attrs,  # Attributes for the containing div
):
    "A vertical group with space between its children"
    return _layout(*c, props={'--gap': gap}, stack=True, **attrs)

None of these children has a margin. The `[stack]` container sets the spacing between them:

In [ ]:
@example(Stack)
def _stack_example():
    return Stack(H2('Account'), P('Manage your profile and preferences.'), Small('Changes apply to this workspace.'))

In [ ]:
Stack.demo()

```html
<div stack>
  <h2>Account</h2>
  <p>Manage your profile and preferences.</p>
<small>Changes apply to this workspace.</small></div>

```

For a tighter gap, pass `gap`. The constructor sets it as `--gap` on the same `div`, with no extra wrapper or styling class:

In [ ]:
to_xml(Stack(P('Ada Lovelace'), Small('Workspace owner'), gap='var(--space-2)'))

'<div stack style="--gap: var(--space-2);">\n  <p>Ada Lovelace</p>\n<small>Workspace owner</small></div>\n'

You can also put `stack=True` on a semantic element. Here the `section` keeps its tag while matching the `[stack]` rule with the default `--space` gap:

In [ ]:
demo(Section(H2('Profile'), P('Visible to the rest of your team.'), stack=True))

```html
<section stack>
  <h2>Profile</h2>
  <p>Visible to the rest of your team.</p>
</section>

```

## Cluster

`Cluster` arranges short items such as tags or metadata in a wrapping flex row. Each child stays at its natural width and aligns centrally on the cross axis. The `--gap` custom property (default `--space`) sets both row and column spacing.

In [ ]:
@hypelt(sel='[cluster]')
def Cluster(
    *c,       # Children to arrange in a wrapping row
    gap=None, # CSS length or token reference; None uses --space
    **attrs,  # Attributes for the containing div
):
    "A row of naturally sized children that wraps to fit"
    return _layout(*c, props={'--gap': gap}, cluster=True, **attrs)

In [ ]:
@example(Cluster)
def _cluster_example():
    return Cluster(Code('FastHTML'), Code('htmx'), Code('semantic HTML'), Code('CSS'), gap='var(--space-2)')

In [ ]:
Cluster.demo()

```html
<div cluster style="--gap: var(--space-2);">
<code>FastHTML</code><code>htmx</code><code>semantic HTML</code><code>CSS</code></div>

```

Inside a narrow container, the same cluster wraps without a media breakpoint even on a wide viewport:

In [ ]:
demo(Div(_cluster_example(), style='max-inline-size: 14rem; border: 1px dashed var(--n-1)'))

```html
<div style="max-inline-size: 14rem; border: 1px dashed var(--n-1)">
  <div cluster style="--gap: var(--space-2);">
<code>FastHTML</code><code>htmx</code><code>semantic HTML</code><code>CSS</code>  </div>
</div>

```

Pass `grow=True` on one child in a `Cluster` to fill the remaining width and push later items to the far edge. Because `[grow]` also sets `min-inline-size: 0`, a long name beside action controls wraps instead of pushing the actions out of the row:

In [ ]:
demo(Cluster(Code('FastHTML'), Stack(P('Ada Lovelace'), Small('Owner'), gap='0', grow=True), Code('Edit'), Code('Remove')))


```html
<div cluster>
<code>FastHTML</code>  <div stack grow style="--gap: 0;">
    <p>Ada Lovelace</p>
<small>Owner</small>  </div>
<code>Edit</code><code>Remove</code></div>

```

## Grid

Unlike `Cluster`, `Grid` places its children in equal-width columns. It fits as many columns as the container allows at a minimum width of `16rem` (`--grid-min`) and distributes the remaining width evenly. When the container is narrower than `--grid-min`, a single column shrinks to fit instead of overflowing. Pass `min_width` to override `--grid-min` without adding media breakpoints.

In [ ]:
@hypelt(sel='[grid]')
def Grid(
    *c,             # Children to arrange in equal-width columns
    gap=None,       # CSS length or token reference; None uses --space
    min_width=None, # Minimum column width; None uses 16rem, capped at the container width
    **attrs,        # Attributes for the containing div
):
    "Equal-width columns whose count adapts to the available width"
    return _layout(*c, props={'--gap': gap, '--grid-min': min_width}, grid=True, **attrs)

Each column contains a stack with a smaller gap. Changing the grid's `gap` doesn't change the stacks inside it, because each layout sets its own `--gap`.

In [ ]:
@example(Grid)
def _grid_example():
    return Grid(
        Stack(H3('Inbox'), P('Requests waiting for a first look.'), gap='var(--space-2)'),
        Stack(H3('In progress'), P('Work the team has picked up.'), gap='var(--space-2)'),
        Stack(H3('Done'), P('Completed work, ready to share.'), gap='var(--space-2)'))

In [ ]:
Grid.demo()

```html
<div grid>
  <div stack style="--gap: var(--space-2);">
    <h3>Inbox</h3>
    <p>Requests waiting for a first look.</p>
  </div>
  <div stack style="--gap: var(--space-2);">
    <h3>In progress</h3>
    <p>Work the team has picked up.</p>
  </div>
  <div stack style="--gap: var(--space-2);">
    <h3>Done</h3>
    <p>Completed work, ready to share.</p>
  </div>
</div>

```

`grow=True` also works inside a `Stack` to fill the remaining height. In a grid of cards, passing `grow=True` on the card body aligns every card's footer to the bottom regardless of title length:

In [ ]:
demo(Grid(Stack(P('A short title'), P('Body', grow=True), Small('Foot')), Stack(P('A title long enough that it wraps onto a second line in a narrow column'), P('Body', grow=True), Small('Foot'))), width='24rem')

```html
<div grid>
  <div stack>
    <p>A short title</p>
    <p grow>Body</p>
<small>Foot</small>  </div>
  <div stack>
    <p>A title long enough that it wraps onto a second line in a narrow column</p>
    <p grow>Body</p>
<small>Foot</small>  </div>
</div>

```

## Lists

Interface groups such as member lists or link menus can still use semantic `Ul` or `Ol` elements. Putting `stack=True`, `cluster=True`, or `grid=True` on a `ul` or `ol` clears the default list markers and padding (`list-style: none; padding: 0`) so its `li` items lay out like any other container.

A stacked `Ul` of names without default list markers:

In [ ]:
demo(Ul(Li('Ada Lovelace'), Li('Grace Hopper'), Li('Alan Turing'), stack=True, cls='gap-2'))

```html
<ul stack style="--gap: var(--space-2)">
  <li>Ada Lovelace</li>
  <li>Grace Hopper</li>
  <li>Alan Turing</li>
</ul>

```

## Flow

`Flow` uses CSS grid with a default row gap of `.5em` (`--flow-space`) for short text sections containing headings and paragraphs. Any heading (`h1` through `h6`) after the first child has an extra `.5em` top margin (`margin-block-start`), putting more space before a heading than after it.

`gap` sets `--flow-space` independently of the `--gap` used by `Stack`, `Cluster`, and `Grid`. You can nest a `Flow` with a smaller `gap` without changing the outer `Flow`'s spacing. For full articles that also style lists, blockquotes, code, and tables, use `prose=True`.

In [ ]:
@hypelt(sel='[flow]')
def Flow(
    *c,       # Headings and text in reading order
    gap=None, # CSS length or token reference; None uses .5em with additional space before headings
    **attrs,  # Attributes for the containing div
):
    "Document flow with closer spacing after headings"
    return _layout(*c, props={'--flow-space': gap}, flow=True, **attrs)

In [ ]:
@example(Flow)
def _flow_example():
    return Flow(H2('Sharing a workspace'), P('Invite your colleagues to work on the same projects.'),
        P('Everyone sees the latest changes.'), H3('Permissions'),
        P('Choose who can edit and who can view.'))

In [ ]:
Flow.demo()

```html
<div flow>
  <h2>Sharing a workspace</h2>
  <p>Invite your colleagues to work on the same projects.</p>
  <p>Everyone sees the latest changes.</p>
  <h3>Permissions</h3>
  <p>Choose who can edit and who can view.</p>
</div>

```

## Composition

The page below uses a stack to space its header, project grid, and help text. The header has its own smaller gap and a cluster of metadata. The article uses flow for its heading and paragraphs. Each layout spaces only its own direct children, whatever layouts its neighbours use.

In [ ]:
demo(Section(
    Header(H2('Workspace'), P('A shared place for the team\'s projects.'),
        Cluster(Small('3 projects'), Small('Updated today'), gap='var(--space-2)'),
        stack=True, cls='gap-2'),
    _grid_example(),
    Article(H3('Working together'), P('Pick a project to see its latest work.'),
        P('Use the inbox for requests that have not yet been assigned.'), flow=True),
    stack=True, cls='gap-8'))

```html
<section stack style="--gap: var(--space2)">
  <header stack style="--gap: var(--space-2)">
    <h2>Workspace</h2>
    <p>A shared place for the team's projects.</p>
    <div cluster style="--gap: var(--space-2);">
<small>3 projects</small><small>Updated today</small>    </div>
  </header>
  <div grid>
    <div stack style="--gap: var(--space-2);">
      <h3>Inbox</h3>
      <p>Requests waiting for a first look.</p>
    </div>
    <div stack style="--gap: var(--space-2);">
      <h3>In progress</h3>
      <p>Work the team has picked up.</p>
    </div>
    <div stack style="--gap: var(--space-2);">
      <h3>Done</h3>
      <p>Completed work, ready to share.</p>
    </div>
  </div>
  <article flow>
    <h3>Working together</h3>
    <p>Pick a project to see its latest work.</p>
    <p>Use the inbox for requests that have not yet been assigned.</p>
  </article>
</section>

```

The outer `Flow` sets a `2rem` gap around the nested `Flow`. The inner `Flow` uses a `.25rem` gap between its two paragraphs:

In [ ]:
demo(Flow(P('Before the nested flow.'), Flow(P('Inner one.'), P('Inner two.'), gap='.25rem'), P('After it.'), gap='2rem'))

```html
<div flow style="--flow-space: 2rem;">
  <p>Before the nested flow.</p>
  <div flow style="--flow-space: .25rem;">
    <p>Inner one.</p>
    <p>Inner two.</p>
  </div>
  <p>After it.</p>
</div>

```

To change a layout default across an application, override its custom property in the `app` CSS layer (for example, `[cluster] { --gap: var(--space-2); }`). For a single element, pass a spacing utility class such as `cls='gap-2'` (`.gap-0`, `.gap-1` through `.gap-4`, `.gap-6`, `.gap-8`, `.gap-12`), pass a constructor parameter such as `gap` for custom lengths, or set the custom property in `style`. Use ordinary CSS or the flex utility classes for other adjustments such as centring a cluster with `justify-content`. None of the layout rules sets a background, border, padding, or outer margin.